In [4]:
#Log in to huggingface hub
from huggingface_hub import login
import os
from dotenv import load_dotenv
load_dotenv()
login(token = os.getenv("HFToken"))
#HFToken = [your token] in .env

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id,padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="cuda:0",dtype=torch.bfloat16)


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [197]:
from datasets import load_dataset
ds = load_dataset("openai/gsm8k", "main")

In [199]:
prompt = [[
    {
        "role": "system",
        "content": "You are a precise calculator. Output ONLY the final numerical answer. Do not include words, units, explanations, or punctuation."
    },
    {"role": "user", "content": "Natalia sold 48 clips in April, and half as many in May. How many clips did she sell in total?"},
    {"role": "assistant", "content": "72"},
    {"role": "user", "content": "Weng earns $12 an hour for babysitting. Yesterday, she babysat for 5 hours. How much money did she earn?"},
    {"role": "assistant", "content": "60"},
    {
        "role": "user",
        "content": question
    },
]for question in ds["train"]["question"]]
input = tokenizer.apply_chat_template(
    prompt,
    tokenize = True,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt",
    clean_up_tokenization_spaces=False
).to("cuda:0")
input_length = input["input_ids"].shape[1]

In [215]:
import json
with open("result.json","w") as f:
    for i in input["input_ids"].unsqueeze(0):
        input_length = len(i)
        output = model.generate(
            input_ids = i,
            max_new_tokens= 10,
            do_sample=True,
            temperature=0.2,
            repetition_penalty=1.2,
            )
        json.dump(tokenizer.decode(output[input_length:],skip_special_tokens=True),f)


[transformers] The attention mask and the pad token id were not set, with a batched input. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


OutOfMemoryError: CUDA out of memory. Tried to allocate 9.84 GiB. GPU 0 has a total capacity of 8.00 GiB of which 2.01 GiB is free. Process 1780 has 17179869184.00 GiB memory in use. Process 3924 has 17179869184.00 GiB memory in use. Process 5800 has 17179869184.00 GiB memory in use. Process 10940 has 17179869184.00 GiB memory in use. Process 10948 has 17179869184.00 GiB memory in use. Process 13232 has 17179869184.00 GiB memory in use. Process 15224 has 17179869184.00 GiB memory in use. Process 11924 has 17179869184.00 GiB memory in use. Process 19504 has 17179869184.00 GiB memory in use. Process 27284 has 17179869184.00 GiB memory in use. Process 22872 has 17179869184.00 GiB memory in use. Process 20600 has 17179869184.00 GiB memory in use. Process 17060 has 17179869184.00 GiB memory in use. Process 11116 has 17179869184.00 GiB memory in use. Process 38356 has 17179869184.00 GiB memory in use. Process 8824 has 17179869184.00 GiB memory in use. Process 7668 has 17179869184.00 GiB memo